# 💧 LFM2.5-VL - SFT with TRL

This tutorial demonstrates how to fine-tune our LFM2.5 vision models [`LiquidAI/LFM2.5-VL-1.6B`](https://huggingface.co/LiquidAI/LFM2.5-VL-1.6B).

Follow along if it's your first time using trl, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **SFT** (Supervised Fine-Tuning) - Basic instruction following
- **LoRA + SFT** - (Optional) using LoRA (from PEFT) to SFT while on constrained hardware

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type`
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages:


In [ ]:
!uv pip install -U transformers peft trl datasets torchvision "torchao>=0.4.0"

Let's now verify the packages are installed correctly


In [ ]:
import torch
import transformers
import trl
import peft
import os
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"📊 PEFT version: {peft.__version__}")

# Loading the model from Transformers 🤗


In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "LiquidAI/LFM2.5-VL-1.6B"

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(
    model_id,
    max_image_tokens=256,
    trust_remote_code=True
)

print("🧠 Loading model...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype="bfloat16",
    device_map="auto",
    trust_remote_code=True
)

print("\n✅ Local model loaded successfully!")
print(f"📖 Vocab size: {len(processor.tokenizer)}")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

SFT teaches the model to follow instructions by training on input-output pairs (instruction vs response). This is the foundation for creating instruction-following models.

Full SFT might be too compute heavy if you're running on one of the free-tier colab GPUs. Hence we use LoRA (Low-Rank Adaptation) to finetune the model by only training a small number of additional parameters. Perfect for limited compute resources!

## Load an SFT Dataset

We will use [simwit/omni-med-vqa-mini](https://huggingface.co/datasets/simwit/omni-med-vqa-mini), which is small dataset for Medical QA.

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("simwit/omni-med-vqa-mini")
full_dataset = raw_ds["test"]
split = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("✅ SFT Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset)}")
print(f"   🧪 Eval samples: {len(eval_dataset)}")
print(f"\n📝 Single Sample: [IMAGE] {train_dataset[0]['question']} {train_dataset[0]['gt_answer']}")

We now transform the dataset to a format compatible with `trl`



In [ ]:
system_message = (
    "You are a medical Vision Language Model specialized in analyzing medical images and providing clinical insights. "
    "Provide concise, clinically relevant answers based on the image and question."
)

def format_medical_sample(sample):
    return [
        {"role": "system", "content": [{"type": "text", "text": system_message}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"].convert("RGB")},
                {"type": "text", "text": sample["question"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["gt_answer"]}]},
    ]

print("✅ SFT dataset formatter ready:")
print(f"   📚 Train samples: {len(train_dataset)}")
print(f"   🧪 Eval samples: {len(eval_dataset)}")

## Collate function
Let's now create a collate function to batch chat text and corresponding RGB images into model-ready tensors using the processor


In [ ]:
def create_collate_fn(processor):
    def collate_fn(samples):
        conversations = [
            format_medical_sample(sample)
            for sample in samples
        ]

        batch = processor.apply_chat_template(
            conversations,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
        return batch

    return collate_fn

collate_fn = create_collate_fn(processor)

## Wrap the model with PEFT (Optional)

We specify target modules that will be finetuned while the rest of the models weights remains frozen. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full-finetuning
- Lower → needs even less compute resources

You can skip this part if you have a premium GPU and want to go for a full finetune.

In [ ]:
from peft import LoraConfig, get_peft_model

target_modules = [
    "q_proj", "v_proj", "fc1", "fc2", "linear",
    "gate_proj", "up_proj", "down_proj",
]

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Launch Training

We are now ready to launch an SFT run with `SFTTrainer`, feel free to modify `SFTConfig` to play around with different configurations.


In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="lfm2-vl-med",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-4,
    warmup_steps=0.1,
    weight_decay=0.01,
    logging_steps=10,
    optim="adamw_torch_8bit",
    gradient_checkpointing=True,
    max_length=512,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
)

print("🏗️  Creating SFT trainer...")
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    processing_class=processor,
)

print("\n🚀 Starting SFT training...")
sft_trainer.train()

print("🎉 SFT training completed!")

sft_trainer.save_model()
print(f"💾 Saving to: {sft_config.output_dir}")

## (Optional) Save merged model

In case you used LoRA, merge the extra weights learned back into the model to obtain a "normal" model checkpoint.

In [ ]:
if hasattr(model, 'peft_config'):
    print("🔄 Merging LoRA weights...")
    model = model.merge_and_unload()
model.save_pretrained("./lfm2-vl-med")
processor.save_pretrained("./lfm2-vl-med")
print("💾 Model saved to: ./lfm2-vl-med")